# 4.2 오픈소스 RL 방법


오픈소스 강화학습 방법들의 특징:
- **접근성**: 학계와 커뮤니티가 개발, 완전 공개
- **성능**: ~5-15% 성공률 (폐쇄소스 대비 낮음)
- **기본 모델**: Llama, Mistral 등 (기본 40% 수준)

주요 방법들:
- **GRPO**: Group Relative Policy Optimization (그룹 비교)
- **PPO**: Proximal Policy Optimization (안정화)
- **DPO**: Direct Preference Optimization (선호도)

**핵심 인사이트**: 성능 격차는 기본 모델과 신호 설계의 차이

### GRPO: 그룹 비교 기반 정책 최적화

절차:
  1) K개의 응답 {y1, ..., yK} 생성 (샘플링)
  2) 각 응답 평가 및 점수 매기기
  3) 상위 k개 평균 vs 하위 k개 평균 비교
  4) 그룹 보상 = mean(top-k) - mean(bottom-k)
  5) 정책 업데이트: L = -log(π(y|x)) × group_reward

장점: 상대적 비교로 신호를 강화


### PPO: 클리핑으로 정책 안정화

문제: 정책이 한 번의 업데이트로 급격히 변할 수 있음
  → 훈련 불안정성, 성능 저하

해결: 클리핑 메커니즘

  $r = π_new(a) / π_old(a)  (확률 비율)$
  
  $Loss = min(r × A, clip(r, 1-ε, 1+ε) × A)$
  
  → 확률 비율을 [1-ε, 1+ε] 범위로 제한
  
  → 정책이 천천히 변함

In [1]:
from utils_openai import *
import numpy as np
from typing import List, Dict

heading("준비: 오픈소스 RL")
print("✓ ask() - LLM으로 응답 생성")
print("✓ llm_reward() - LLM으로 보상 평가")
print("✓ MemoryStream - 학습 궤적 추적")


────────────────────────────────────────
  준비: 오픈소스 RL
────────────────────────────────────────
✓ ask() - LLM으로 응답 생성
✓ llm_reward() - LLM으로 보상 평가
✓ MemoryStream - 학습 궤적 추적


## 실습 1: GRPO 그룹 비교

In [2]:
heading("실습 1: GRPO 그룹 비교")

def simulate_grpo(num_candidates=6):
    """여러 응답을 생성하고 그룹 비교 보상을 계산한다."""
    
    # 시뮬레이션: 6개 응답 생성 및 평가
    scores = []
    for i in range(num_candidates):
        # 각 응답의 품질 점수
        score = 0.4 + np.random.uniform(0, 0.5)
        scores.append(score)
    
    scores.sort()
    
    # 상위 k개와 하위 k개 분리
    k = 2
    top_k = scores[-k:]
    bottom_k = scores[:k]
    
    top_mean = np.mean(top_k)
    bottom_mean = np.mean(bottom_k)
    group_reward = top_mean - bottom_mean
    
    return {"all_scores": scores, "top_mean": top_mean, 
            "bottom_mean": bottom_mean, "group_reward": group_reward}

result = simulate_grpo()

step_print(1, "응답 평가", f"6개 응답")
print(f"  점수: {[f'{s:.2f}' for s in result['all_scores']]}")

step_print(2, "그룹 보상 계산", "상위 vs 하위")
print(f"  상위 2개 평균: {result['top_mean']:.3f}")
print(f"  하위 2개 평균: {result['bottom_mean']:.3f}")
print(f"  그룹 보상: {result['group_reward']:.3f}")
print(f"  → 정책이 이 방향으로 업데이트된다")
print(f"  상위 응답을 더 자주 생성하도록 학습")


────────────────────────────────────────
  실습 1: GRPO 그룹 비교
────────────────────────────────────────
  [Step 1] 응답 평가
    6개 응답
  점수: ['0.72', '0.78', '0.82', '0.84', '0.87', '0.89']
  [Step 2] 그룹 보상 계산
    상위 vs 하위
  상위 2개 평균: 0.882
  하위 2개 평균: 0.751
  그룹 보상: 0.132
  → 정책이 이 방향으로 업데이트된다
  상위 응답을 더 자주 생성하도록 학습


## 실습 2: PPO 클리핑 메커니즘

In [4]:
heading("실습 2: PPO 클리핑")

def simulate_ppo_clipping(prob_ratio, advantage, epsilon=0.2):
    """PPO 클리핑의 효과를 보여준다."""
    
    # 클리핑 없는 손실
    unclipped_loss = prob_ratio * advantage
    
    # 클리핑된 손실
    clipped_ratio = np.clip(prob_ratio, 1-epsilon, 1+epsilon)
    clipped_loss = clipped_ratio * advantage
    
    # 최종 손실 (둘 중 작은 값)
    final_loss = min(unclipped_loss, clipped_loss)
    
    return {
        "prob_ratio": prob_ratio,
        "unclipped": unclipped_loss,
        "clipped": clipped_loss,
        "final": final_loss,
        "is_clipped": clipped_loss != unclipped_loss
    }

step_print(1, "PPO 클리핑 시나리오", "극단적 정책 변화 방지")

scenarios = [
    {"name": "정상 업데이트", "ratio": 1.1, "adv": 1.0},
    {"name": "과도한 업데이트", "ratio": 2.0, "adv": 1.0},
    {"name": "극단적 업데이트", "ratio": 3.0, "adv": 1.0}
]

for scenario in scenarios:
    result = simulate_ppo_clipping(scenario["ratio"], scenario["adv"])
    print(f"    [{scenario['name']}]")
    print(f"    확률 비율: {result['prob_ratio']:.2f}")
    print(f"    클리핑됨: {'네' if result['is_clipped'] else '아니오'}")
    print(f"    최종 손실: {result['final']:.3f}")


────────────────────────────────────────
  실습 2: PPO 클리핑
────────────────────────────────────────
  [Step 1] PPO 클리핑 시나리오
    극단적 정책 변화 방지
    [정상 업데이트]
    확률 비율: 1.10
    클리핑됨: 아니오
    최종 손실: 1.100
    [과도한 업데이트]
    확률 비율: 2.00
    클리핑됨: 네
    최종 손실: 1.200
    [극단적 업데이트]
    확률 비율: 3.00
    클리핑됨: 네
    최종 손실: 1.200


## 실습 3: 학습 곡선 추적

In [5]:
heading("실습 3: 오픈소스 RL 학습 곡선")

# 오픈소스 RL의 전형적 학습 곡선
epochs = 20
rewards = []
current_reward = 0.05  # 초기: 5% 성공률

for epoch in range(1, epochs + 1):
    # 로그 함수로 감소하는 증가율
    improvement = 0.10 * np.log(epoch + 1) / np.log(epochs + 1)
    current_reward = 0.05 + improvement
    rewards.append(current_reward)

step_print(1, "학습 진행", f"{epochs}개 에포크")

for epoch, reward in enumerate(rewards, 1):
    if epoch % 5 == 1 or epoch == 1:
        bar = "█" * int(reward * 30)
        print(f"  에포크 {epoch:2d}: {bar} {reward:.1%}")

print(f"  초기 성공률: {rewards[0]:.1%}")
print(f"  최종 성공률: {rewards[-1]:.1%}")
print(f"  개선: {(rewards[-1] - rewards[0]):.1%}")


────────────────────────────────────────
  실습 3: 오픈소스 RL 학습 곡선
────────────────────────────────────────
  [Step 1] 학습 진행
    20개 에포크
  에포크  1: ██ 7.3%
  에포크  6: ███ 11.4%
  에포크 11: ███ 13.2%
  에포크 16: ████ 14.3%
  초기 성공률: 7.3%
  최종 성공률: 15.0%
  개선: 7.7%


## 요약: 오픈소스 vs 폐쇄소스

### 성능 비교

| 항목 | 오픈소스 | 폐쇄소스 |
|------|--------|--------|
| 기본 모델 성능 | ~40% | ~70% |
| RL 후 성능 | ~5-15% | ~51.5% |
| API 비용 | 낮음 | 높음 |
| 접근성 | 완전 공개 | 제한됨 |
| 신호 설계 | 기본적 | 정교함 |

### 핵심 통찰

1. **기본 모델이 상한선을 결정한다**
   - 기본 모델 성능이 높을수록 RL 성능도 높다
   - GPT-4o(70%) vs Llama(40%)의 차이 → 성능 격차 발생

2. **GRPO는 상대적 신호로 학습을 강화한다**
   - 절대적 보상보다 상대적 비교가 더 효율적
   - 응답 간 차이를 극대화하는 방향으로 정책 개선

3. **PPO는 안정성을 제공한다**
   - 클리핑으로 극단적 정책 변화를 방지
   - 훈련 곡선이 더 안정적이고 예측 가능

4. **성능 격차는 개선 가능하다**
   - 더 나은 신호 설계
   - 더 큰 규모의 기본 모델 사용
   - 더 많은 훈련 데이터